# Recipe 3 — Find Tracks Overlapping a Genomic Region

This notebook walks through querying the FILER coordinate-overlap endpoint to find all
tracks whose intervals intersect a genomic region of interest.

**What you'll learn:**
- How to query the FILER coordinate-overlap endpoint
- How to use `count_only` vs full metadata modes
- How to apply `filterString` to narrow results by metadata
- How to explore and summarize overlapping tracks
- How to save results for downstream workflows

**Use when:** you have a specific locus of interest (e.g., a GWAS hit, a candidate enhancer)
and want to discover which FILER tracks have signal there.

**Next steps after this notebook:**
- `01_track_discovery.ipynb` — pre-filter the track universe before querying
- `02_query_and_summarize.ipynb` — batch-query a BED file of regions

Run the following before starting:
```
pip install -e ".[dev]"
```

---
## 0. Setup

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

In [ ]:
from filerpy.client import get_overlapping_tracks, ENDPOINTS

In [ ]:
print(f"FILER coordinate-overlap endpoint: {ENDPOINTS['region_overlaps']}")

---
## 1. Hello World — count tracks overlapping a region

The fastest way to verify your connection is to request overlap counts only (`countOnly=1`).
This returns just the track `identifier` and `num_overlaps` — no full metadata fetched.

In [ ]:
results = get_overlapping_tracks(
    "chr1:100000-200000",
    "hg38",
    countOnly=1,
)
print(f"Tracks with overlaps: {len(results)}")
print(json.dumps(results[:3], indent=2))

---
## 2. Configure your query

Edit the parameters below to match your locus of interest.

| Parameter | PHP param | Notes |
|---|---|---|
| `region` | `region` | Format: `chrN:start-end` |
| `genome_build` | `genomeBuild` | `hg19`, `hg38`, or `hg38-lifted` |
| `count_only` | `countOnly` | `1` = counts only (fast), `0` = include download URLs |
| `full_metadata` | `fullMetadata` | `1` = all 36 metadata columns; **required for `filterString`** |
| `filter_string` | `filterString` | jq-style boolean expression; requires `full_metadata=1` |

> ⚠️ `filterString` requires `fullMetadata=1`. Without it, metadata fields are absent and
> every filter evaluates to `null == "value"`, returning empty results.

In [ ]:
# ── Edit these parameters to match your use case ───────────────────────────
REGION        = "chr1:100000-200000"  # required
GENOME_BUILD  = "hg38"               # required: "hg19", "hg38", or "hg38-lifted"
COUNT_ONLY    = 0                    # 0 = full results, 1 = counts only (fastest)
FULL_METADATA = 1                    # 1 = all metadata columns; required for filterString
FILTER_STRING = None                 # e.g. '.data_source == "ENCODE"', or None for all
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
# Auto-enable full_metadata when a filter is provided
if FILTER_STRING:
    FULL_METADATA = 1
    COUNT_ONLY = 0
    print("filterString detected — forcing full_metadata=1 and count_only=0")

In [ ]:
print(f"Region        : {REGION}")
print(f"Genome build  : {GENOME_BUILD}")
print(f"Count only    : {COUNT_ONLY}")
print(f"Full metadata : {FULL_METADATA}")
print(f"Filter string : {FILTER_STRING}")

---
## 3. Run the query

In [ ]:
print(f"Querying FILER for region: {REGION} ({GENOME_BUILD})...")

In [ ]:
data = get_overlapping_tracks(
    REGION,
    GENOME_BUILD,
    countOnly=COUNT_ONLY,
    fullMetadata=FULL_METADATA,
    **({"filterString": FILTER_STRING} if FILTER_STRING else {}),
)
df = pd.DataFrame(data)

In [ ]:
print(f"\nTracks with overlaps: {len(df)}")
print(df.head().to_string())

### 3a. View key columns only

In [ ]:
# Adjust KEY_COLS depending on what was returned
available_cols = df.columns.tolist()

In [ ]:
PREFERRED_COLS = [
    "identifier", "assay", "cell_type", "tissue_category",
    "life_stage", "data_source", "output_type", "track_name", "num_overlaps",
]
KEY_COLS = [c for c in PREFERRED_COLS if c in available_cols]

In [ ]:
print(f"Displaying columns: {KEY_COLS}")
print(df[KEY_COLS].head(10).to_string())

### 3b. Advanced: raw jq `filterString` examples

For power users — pass a raw jq expression for full Boolean flexibility.
Re-run section 2 and 3 after changing `FILTER_STRING`.

```python
# Single condition
FILTER_STRING = '.data_source == "ENCODE"'

# Multiple AND conditions
FILTER_STRING = '.data_source == "ENCODE" and .assay == "ATAC-seq" and .tissue_category == "Blood" and .life_stage == "Adult"'

# OR across data sources
FILTER_STRING = '(.data_source == "ENCODE" or .data_source == "Blueprint") and .output_type == "peaks"'

# No filter — return all overlapping tracks
FILTER_STRING = None
```

---
## 4. Explore the results

Before downstream use, understand the shape of your results.
*(Cells below are skipped automatically if metadata columns are absent.)*

In [ ]:
if "data_source" in df.columns:
    print("=== Tracks by data_source ===")
    print(df["data_source"].value_counts().to_string())
else:
    print("data_source not available — re-run with full_metadata=1")

In [ ]:
if "assay" in df.columns:
    print("=== Tracks by assay ===")
    print(df["assay"].value_counts().to_string())
else:
    print("assay not available — re-run with full_metadata=1")

In [ ]:
if "tissue_category" in df.columns:
    print("=== Tracks by tissue_category ===")
    print(df["tissue_category"].value_counts().to_string())
else:
    print("tissue_category not available — re-run with full_metadata=1")

In [ ]:
if "num_overlaps" in df.columns:
    print("=== num_overlaps distribution ===")
    print(df["num_overlaps"].describe())

In [ ]:
# Bar chart — tracks per data source (only when metadata is available)
if "data_source" in df.columns:
    counts = df["data_source"].value_counts()
    fig, ax = plt.subplots(figsize=(8, max(3, len(counts) * 0.4)))
    counts.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("Number of tracks")
    ax.set_title(f"FILER overlapping tracks by data source\n({REGION}, {GENOME_BUILD})")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — data_source not available")

In [ ]:
# Bar chart — tracks per assay
if "assay" in df.columns:
    assay_counts = df["assay"].value_counts()
    fig, ax = plt.subplots(figsize=(8, max(3, len(assay_counts) * 0.4)))
    assay_counts.sort_values().plot.barh(ax=ax, color="orchid")
    ax.set_xlabel("Number of tracks")
    ax.set_ylabel("Assay")
    ax.set_title(f"FILER overlapping tracks by assay\n({REGION}, {GENOME_BUILD})")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — assay not available")

---
## 5. Refine your selection

Post-query filtering in pandas — useful for narrowing results without re-querying the API.

In [ ]:
if "life_stage" in df.columns and "num_overlaps" in df.columns:
    # ── Edit these conditions to match what you need ────────────────────────
    df_filtered = df[
        (df["life_stage"] == "Adult") &
        (df["num_overlaps"] >= 1)
    ].copy()
    # ─────────────────────────────────────────────────────────────────────────

    print(f"After refinement: {len(df_filtered)} tracks (from {len(df)} total)")
    print(df_filtered[KEY_COLS].head().to_string())
else:
    df_filtered = df.copy()
    print("Metadata columns not available for refinement — using full result set")

In [ ]:
# Sanity check: are all identifiers unique?
if "identifier" in df_filtered.columns:
    n_dupes = df_filtered["identifier"].duplicated().sum()
    print(f"Duplicate identifiers: {n_dupes}")
    if n_dupes > 0:
        print("Deduplicating...")
        df_filtered = df_filtered.drop_duplicates("identifier")

---
## 6. Save results

### 6a. Save as TSV

In [ ]:
repo_root = Path().resolve().parents[1]
out_dir = repo_root / "output" / "03-coordinate-search"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
tsv_path = out_dir / "search_results.tsv"
df_filtered.to_csv(tsv_path, sep="\t", index=False)
print(f"Saved {len(df_filtered)} tracks → {tsv_path}")

### 6b. Save query provenance (recommended for reproducibility)

Saves the exact parameters used alongside results so the query is fully reproducible.

In [ ]:
provenance = {
    "query": {
        "region":        REGION,
        "genome_build":  GENOME_BUILD,
        "count_only":    COUNT_ONLY,
        "full_metadata": FULL_METADATA,
        "filter_string": FILTER_STRING,
    },
    "results": {
        "total_overlapping": len(df),
        "after_refinement":  len(df_filtered),
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "endpoint":  ENDPOINTS["region_overlaps"],
}

In [ ]:
prov_path = out_dir / "search_provenance.json"
with open(prov_path, "w") as f:
    json.dump(provenance, f, indent=2)

In [ ]:
print(f"Provenance saved → {prov_path}")
print(json.dumps(provenance, indent=2))

---
## 7. Batch queries across multiple regions

If you need overlaps for multiple loci, loop and concatenate.
For large BED files, use Recipe 3.2 (batch-query script) instead. 
Please refrain from inputting too many regions as this would overload the server

In [ ]:
regions_of_interest = [
    "chr1:100000-200000",
    "chr2:500000-600000",
    "chr3:1000000-1100000",
]

In [ ]:
frames = []
for region in regions_of_interest:
    data = get_overlapping_tracks(region, GENOME_BUILD, countOnly=1)
    chunk = pd.DataFrame(data)
    chunk["query_region"] = region
    frames.append(chunk)
    print(f"  {region} → {len(chunk)} overlapping tracks")

In [ ]:
df_batch = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows across all regions: {len(df_batch)}")
print(df_batch.head().to_string())

---
## 8. Next steps

You now have a `search_results.tsv` and `search_provenance.json` ready for downstream use.

| What to do next | Recipe / Notebook |
|---|---|
| Pre-filter the track universe before querying | Recipe 1 / `01_track_discovery.ipynb` |
| Batch-query a BED file of regions | Recipe 3.2 |
| Summarize overlaps by assay / cell type | Recipe 4.1 |
| Deploy tracks locally for fast offline queries | Recipe 2.2 |

```python
# Quick reference: reload your saved results in a future session
import pandas as pd
df = pd.read_csv("output/03-coordinate-search/search_results.tsv", sep="\t")
print(df.head())
```